# GeoQuery STAC API — Demo

A living walkthrough of every endpoint under `/api/stac/v1/`, GeoQuery's
STAC (SpatioTemporal Asset Catalog) discovery API. Re-run this notebook
against a running GeoQuery dev server to see the API respond for real —
as endpoints are added or change shape, update the matching cell here
rather than letting this drift out of sync.

For the full, always-current reference, see the Swagger UI at
`/api/stac/v1/docs/` and the raw OpenAPI schema at `/api/stac/v1/schema/`
— this notebook is the narrated, run-it-yourself companion to those, not
a replacement.

## 1. Setup

Assumes the GeoQuery dev stack is running locally (`docker compose up -d`)
so the backend is reachable at `localhost:8000`. Point `BASE_URL` elsewhere
to hit a different environment.

In [ ]:
%pip install requests pystac-client -q

import json

import requests

BASE_URL = "http://localhost:8000/api/stac/v1"


def call(method, path, **kwargs):
    """Make a request and print status + pretty-printed JSON body."""
    response = requests.request(method, f"{BASE_URL}{path}", **kwargs)
    print(f"{method} {path} -> {response.status_code}")
    try:
        data = response.json()
        print(json.dumps(data, indent=2)[:2000])
        return data
    except ValueError:
        print(response.text[:500])
        return None

## 2. Landing page

`GET /` — the STAC Catalog / OGC API landing page. Lists the catalog's
`conformsTo` classes and the `links` a client follows to discover
collections, search, and documentation.

In [ ]:
landing = call("GET", "/")
print(f"\nconforms to {len(landing['conformsTo'])} classes")
print("links:", [link["rel"] for link in landing["links"]])

## 3. Conformance

`GET /conformance/` — the same `conformsTo` list as the landing page, on
its own endpoint. This is how a client checks what a catalog supports
before trying to use it (e.g. "does this catalog support Item Search?").

In [ ]:
call("GET", "/conformance/")

## 4. Browse collections

`GET /collections/` — every active, public Dataset *and* FeatureCollection
(boundary set) in GeoQuery, as STAC Collections. Both kinds are mixed
together in one list — there's no separate "boundaries" endpoint here,
unlike the GeoQuery-native public API.

In [ ]:
collections_response = call("GET", "/collections/")
collections = collections_response["collections"]
print(f"\n{len(collections)} collection(s) available")

## 5. Get a collection by id

`GET /collections/{name}/` — full detail for one collection, including its
spatial/temporal extent and (for boundary sets) `summaries`.

In [ ]:
if collections:
    collection_id = collections[0]["id"]
    call("GET", f"/collections/{collection_id}/")
else:
    print("No collections in this environment yet — skipping.")

## 6. Browse items in a collection

`GET /collections/{name}/items/` — the Items under a collection. For a
dataset-backed collection this is one Item per `DatasetResource` (e.g.
one per year of a time series), paginated via `limit`/`offset` with a
`next` link when more remain.

In [ ]:
# Collections without "summaries" are Dataset-backed (only boundary sets
# get summaries) — pick one of those specifically so the items walkthrough
# below shows real DatasetResource items, not the single synthetic one.
dataset_collection = next((c for c in collections if "summaries" not in c), None)
if dataset_collection:
    dataset_collection_id = dataset_collection["id"]
    items_response = call("GET", f"/collections/{dataset_collection_id}/items/")
    items = items_response["features"]
    print(
        f"\n{items_response['numberMatched']} matched, "
        f"{items_response['numberReturned']} returned"
    )
else:
    items = []
    dataset_collection_id = None
    print("No dataset-backed collections in this environment yet — skipping.")

## 7. Get an item by id

`GET /collections/{name}/items/{item_id}/` — a single Item. The `via`
link is how a discoverer gets from metadata to actually requesting this
data through GeoQuery — this API deliberately doesn't serve raw files
itself (see the design spec's "no downloadable assets" decision).

In [ ]:
if items:
    item_id = items[0]["id"]
    item = call("GET", f"/collections/{dataset_collection_id}/items/{item_id}/")
    via_link = next(link for link in item["links"] if link["rel"] == "via")
    print(f"\nvia: {via_link['href']}")
else:
    item = None
    print("No items in this environment yet — skipping.")

## 8. A boundary set's synthetic item

Boundaries (`FeatureCollection`s) are Collections too, but they get
exactly *one* Item each, not one per administrative unit — a boundary
set is distributed as a single file, not a time series, and individual
`Feature` rows carry no metadata beyond geometry. Its id is always
`{collection_id}-item`.

In [ ]:
# The mirror image of the dataset-collection pick above: only boundary
# sets carry a "summaries" key, so this is how we find one specifically.
boundary_collection = next((c for c in collections if "summaries" in c), None)
if boundary_collection:
    boundary_id = boundary_collection["id"]
    boundary_items = call("GET", f"/collections/{boundary_id}/items/")
    print(f"\n{len(boundary_items['features'])} item(s) — should always be exactly 1")
    print("item id:", boundary_items["features"][0]["id"])
else:
    print("No boundary-backed collections in this environment yet — skipping.")

## 9. Search

`GET`/`POST /search/` — cross-collection Item search, filtered by `bbox`,
`datetime` (RFC3339, `start/end`, or `..` for open-ended), `collections`,
and `limit`. Both GET (query params) and POST (JSON body) accept the same
filters and return the same shape.

In [ ]:
all_items = call("GET", "/search/")
print(f"\n{all_items['numberMatched']} item(s) across the whole catalog")

In [ ]:
if item and item.get("bbox"):
    xmin, ymin, xmax, ymax = item["bbox"]
    bbox_param = f"{xmin},{ymin},{xmax},{ymax}"
    filtered = call(
        "GET",
        "/search/",
        params={"bbox": bbox_param, "collections": dataset_collection_id},
    )
    print(
        f"\n{filtered['numberMatched']} item(s) matching bbox={bbox_param} "
        f"in collection {dataset_collection_id}"
    )
else:
    filtered = None
    print("No item bbox available to filter on — skipping.")

In [ ]:
if item and item.get("bbox"):
    post_filtered = call(
        "POST",
        "/search/",
        json={"bbox": item["bbox"], "collections": [dataset_collection_id]},
    )
    print(f"\nGET and POST agree: {filtered['numberMatched'] == post_filtered['numberMatched']}")
else:
    print("Skipped — no bbox available (see previous cell).")

## 10. Real STAC client interop

Everything above used raw `requests` to show the exact JSON shape. This
section instead uses `pystac-client` — a real, independently-maintained
STAC client library — talking to this catalog. A hand-rolled server is
exactly as interoperable as a purpose-built one as long as it's genuinely
spec-conformant, which is what this section demonstrates concretely.

In [ ]:
from pystac_client import Client

catalog = Client.open(BASE_URL)
print(f"Opened catalog: {catalog.title}")

pystac_collections = list(catalog.get_collections())
print(f"\n{len(pystac_collections)} collection(s) via pystac-client")
if pystac_collections:
    print("first collection:", pystac_collections[0].id, "-", pystac_collections[0].title)

In [ ]:
search = catalog.search(
    collections=[dataset_collection_id] if dataset_collection_id else None,
    max_items=5,
)
found_items = list(search.item_collection())
print(f"{len(found_items)} item(s) via pystac-client search")
for found_item in found_items:
    print(" -", found_item.id, found_item.datetime)

## Keeping this notebook current

When an endpoint is added, removed, or changes shape in `backend/stac_api/`,
update the matching section above in the same PR — treat this notebook as
part of the STAC API's contract, not an afterthought. `/api/stac/v1/schema/`
is the source of truth if this notebook and the real API ever disagree.